### analysis of dataset 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
#Load and inspect the data
 
df = pd.read_csv("data/SAMPL.csv")
 
# Keep only the columns we need
df = df[['smiles', 'expt']].copy()
 
# Drop missing values
df = df.dropna(subset=['smiles', 'expt']).reset_index(drop=True)
 
print("Dataset shape:", df.shape)
print(df.head())
print() 

In [ ]:
# Descriptive Statistics

print(df[['expt']].describe())
print()

plt.figure(figsize=(7, 5))
plt.hist(df['expt'], bins=30, edgecolor='black')
plt.xlabel("Experimental ΔG_solv (kcal/mol)")
plt.ylabel("Frequency")
plt.title("Distribution of Experimental Hydration Free Energies")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Convert SMILES to Morgan Fingerprints

# Morgan fingerprints encode molecular structure as a binary vector.
# Each bit represents whether a particular substructure (circular
# environment of radius 2 around each atom) is present (1) or absent (0).
#
# Parameters:
#   radius=2   : consider atom environments up to 2 bonds away
#   fpSize=2048: the fingerprint has 2048 bits (features)
 
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
 
def smiles_to_morgan_fp(smiles):
    """Convert a SMILES string to a Morgan fingerprint (numpy array)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = morgan_gen.GetFingerprintAsNumPy(mol)
    return fp
 
# Convert all molecules
X_list = []
y_list = []
 
for _, row in df.iterrows():
    fp = smiles_to_morgan_fp(row["smiles"])
    if fp is not None:
        X_list.append(fp)
        y_list.append(row["expt"])
 
X = np.array(X_list) #molecular descriptor 
y = np.array(y_list) #true experimental solvation energies
 
print(f"X shape: {X.shape}  (molecules x fingerprint bits)") #predictor matrix
print(f"y shape: {y.shape}  (experimental dG_solv values)") #target vector
print()

In [ ]:
#Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
 
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples : {X_test.shape[0]}")
print()

In [ ]:
# Train Linear Regression

model = LinearRegression()
model.fit(X_train, y_train)
 

print(f"Number of features (weights): {len(model.coef_)}")
print(f"Intercept: {model.intercept_:.4f} kcal/mol")
print()

In [ ]:
#Predict on Test Set and Evaluate

#predict the experimental solvation free energy for the molecules in the test set
y_pred = model.predict(X_test) #x bits are features

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
 
print(f"RMSE: {rmse:.4f} kcal/mol")
print(f"MAE : {mae:.4f} kcal/mol")
print(f"R^2 : {r2:.4f}")
print()
 
# Show sample predictions
results = pd.DataFrame({
    "Actual_expt": y_test,
    "Predicted_expt": y_pred
})
print("Sample predictions:")
print(results.head(10))
print()

In [ ]:
#Actual vs Predicted Plot

residuals = y_test - y_pred #true experimental value and the predicted value
 
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
 
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min_val, max_val], [min_val, max_val], '--', color = 'coral', linewidth=2)
plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("Linear Regression: Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
 

This scatter plot compares the true experimental solvation free energies with the values predicted by the linear regression model. The dashed diagonal line represents perfect prediction: points lying close to this line are predicted accurately. Most points follow the general diagonal trend, showing that the model captures an overall relationship between molecular structure and expt, but the spread around the line indicates prediction errors, with some clear outliers.

In [ ]:
#Residual Plot
 
plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(y=0, linestyle='--', color = 'coral', linewidth=2)
plt.xlabel("Predicted expt")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot: Linear Regression")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#Residual Distribution
 
plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=20, edgecolor="black")
plt.axvline(x=0, linestyle='--', color = 'coral', linewidth=2)
plt.xlabel("Residuals (Actual - Predicted)")
plt.ylabel("Frequency")
plt.title("Distribution of Residuals")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

This **plot shows the residuals** (Actual − Predicted) against the predicted experimental values. A good model should produce residuals randomly scattered around zero with no strong pattern. Here, most residuals are centred near zero, which suggests the model is reasonable overall, but several large positive and negative residuals are present, indicating molecules that are predicted poorly. The more extreme residuals suggest that the model struggles for some compounds.

This **histogram** shows the distribution of the residuals. Most residuals are concentrated around zero, which means many predictions are fairly close to the experimental values. However, the distribution has some long tails, especially on the negative side, showing that a few molecules have much larger errors. This suggests that while the model performs moderately well for many cases, it does not predict all molecules equally accurately.

In [ ]:
#Identify Outliers

outlier_df = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred,
    "residual": residuals,
    "abs_residual": np.abs(residuals)
})

outlier_df = outlier_df.sort_values("abs_residual", ascending=False)
 
print(outlier_df.head(10).to_string())
print()

In [ ]:
# Visualisation of Outliers

# Get top 10 outliers
top_outliers = outlier_df.head(10)

# Plot 1: Actual vs Predicted with outliers highlighted
plt.figure(figsize=(7, 7))

plt.scatter(y_test, y_pred, alpha=0.5, label="Test data")
plt.scatter(top_outliers["actual"], top_outliers["predicted"],color="orange", s=80, edgecolors="black", zorder=5, label="Top 10 outliers")
plt.plot([min_val, max_val], [min_val, max_val], '--', linewidth=2, color='gray')

plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("Outliers Highlighted on Actual vs Predicted")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The highlighted outliers are the molecules with the largest prediction errors. Because they are far from the diagonal line, they represent cases where the model does not reproduce the experimental solvation free energy well. These points suggest that some molecules have structural or chemical properties that are not adequately captured by the fingerprint-based linear regression model.

In [ ]:
#Final Summary
 
y_pred_train = model.predict(X_train)
r2_train = r2_score(y_train, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
 

print(f"Dataset: SAMPL hydration free energies ({len(y)} molecules)")
print(f"Features: Morgan fingerprints (radius=2, {X.shape[1]} bits)")
print(f"Model: Linear Regression")
print()
print(f"Training set ({X_train.shape[0]} molecules):")
print(f"  R^2  = {r2_train:.4f}")
print(f"  RMSE = {rmse_train:.4f} kcal/mol")
print()
print(f"Test set ({X_test.shape[0]} molecules):")
print(f"  R^2  = {r2:.4f}")
print(f"  RMSE = {rmse:.4f} kcal/mol")
print(f"  MAE  = {mae:.4f} kcal/mol")

In [ ]:
# Physics/computed baseline: calc vs experimental expt

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

df_full = pd.read_csv("data/SAMPL.csv")

print("Columns in data/SAMPL.csv:")
print(df_full.columns.tolist())

calc_candidates = [c for c in df_full.columns if "calc" in c.lower()]

if len(calc_candidates) == 0:
    print("\nNo 'calc' column was found.")
    print("Skipping the calc-vs-expt baseline because no computed column is available.")
else:
    calc_col = calc_candidates[0]
    print(f"\nUsing calculated column: {calc_col}")

    df_calc = df_full[["smiles", "expt", calc_col]].dropna().reset_index(drop=True)

    y_true_calc = df_calc["expt"].to_numpy(dtype=float)
    y_pred_calc = df_calc[calc_col].to_numpy(dtype=float)

    r2_calc = r2_score(y_true_calc, y_pred_calc)
    rmse_calc = np.sqrt(mean_squared_error(y_true_calc, y_pred_calc))
    mae_calc = mean_absolute_error(y_true_calc, y_pred_calc)

    print("\nComputed/physics-based baseline on all available SAMPL molecules:")
    print(f"R²   = {r2_calc:.4f}")
    print(f"RMSE = {rmse_calc:.4f} kcal/mol")
    print(f"MAE  = {mae_calc:.4f} kcal/mol")

## Connection to Solvation Theory

### What we did
We built a **QSPR (Quantitative Structure-Property Relationship)** model that predicts the experimental hydration free energy (ΔG_solv) directly from molecular structure. The pipeline was:

**SMILES → Morgan Fingerprints (2048 bits) → Linear Regression → Predicted ΔG_solv**

This is a purely **data-driven** approach: the model learns which structural features (substructures encoded in the fingerprint bits) correlate with favourable or unfavourable solvation, without any physics-based calculation.


### The three strategies in context

| Approach | What it uses | Typical accuracy | Our result |
|----------|-------------|-----------------|------------|
| **Explicit solvation** (Strategy 1) | Every solvent molecule simulated | High (if converged) | Not tested here |
| **Continuum solvation** (Strategy 2) | Physics-based ΔG_solv calculation | 1–2 kcal/mol (neutrals) | `calc` column: R² = 0.84, RMSE = 1.54 kcal/mol (computed here) |
| **QSPR / Data-driven** | Molecular descriptors + regression | Variable | **R² = 0.71, RMSE = 2.2 kcal/mol** |
| **Hybrid** (Strategy 3) | Explicit 1st shell + continuum bulk | 3–5 kcal/mol (ions) | Not tested here |

Our fingerprint-based linear regression is a QSPR model. The `calc` column in the original dataset represents a physics-based computed hydration free energy baseline. In this notebook, it achieves R² = 0.839, RMSE = 1.542 kcal/mol and MAE = 1.114 kcal/mol, while the fingerprint-based Linear Regression model achieves R² = 0.708 and RMSE = 2.199 kcal/mol on the test split. This shows that the computed physics-based baseline outperforms the simple fingerprint-only linear model, likely because it encodes physical information about solvation that the fingerprint model must learn from limited data.

### What could improve the model
- **Regularisation** (Ridge or Lasso regression) to reduce overfitting with 2048 features
- **Fewer, more informative descriptors** instead of raw fingerprints (e.g. molecular weight, polar surface area, log P)
- **Combining physics + data**: using the computed ΔG_solv (`calc`) as an additional feature alongside fingerprints — the hybrid approach that McDonagh et al. tested in Skyner et al.'s review
- **More training data**: the SAMPL dataset has only 642 molecules; larger datasets would better support high-dimensional features

### ridge, lasso and sgd

In [ ]:
#Ridge regression

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv("data/SAMPL.csv")
df = df[["smiles", "expt"]].dropna().reset_index(drop=True)

#Convert SMILES to Morgan fingerprints
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprintAsNumPy(mol)

X_list = []
y_list = []

for _, row in df.iterrows():
    fp = smiles_to_morgan_fp(row["smiles"])
    if fp is not None:
        X_list.append(fp)
        y_list.append(row["expt"])

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)

#Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Ridge Regression (best alpha = 1.0)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

y_train_pred_ridge = ridge_model.predict(X_train)
y_test_pred_ridge = ridge_model.predict(X_test)

ridge_train_r2 = r2_score(y_train, y_train_pred_ridge)
ridge_test_r2 = r2_score(y_test, y_test_pred_ridge)
ridge_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))
ridge_test_mae = mean_absolute_error(y_test, y_test_pred_ridge)

print("\n RIDGE REGRESSION (alpha = 1.0)")
print(f"Training R²: {ridge_train_r2:.4f}")
print(f"Test R²: {ridge_test_r2:.4f}")
print(f"Training RMSE: {ridge_train_rmse:.4f}")
print(f"Test RMSE: {ridge_test_rmse:.4f}")
print(f"Test MAE: {ridge_test_mae:.4f}")

# Ridge: Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred_ridge, alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "--", color = 'orange', linewidth=2)
plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("Ridge Regression (alpha = 1.0): Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.show()

# Ridge: Residual plot
ridge_residuals = y_test - y_test_pred_ridge

plt.figure(figsize=(8, 6))
plt.scatter(y_test_pred_ridge, ridge_residuals, alpha=0.7)
plt.axhline(0, linestyle="--", color = 'orange', linewidth=2)
plt.xlabel("Predicted expt")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Ridge Regression (alpha = 1.0): Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Lasso regression

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


#Load dataset
df = pd.read_csv("data/SAMPL.csv")
df = df[["smiles", "expt"]].dropna().reset_index(drop=True)

#Convert SMILES to Morgan fingerprints
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprintAsNumPy(mol)

X_list = []
y_list = []

for _, row in df.iterrows():
    fp = smiles_to_morgan_fp(row["smiles"])
    if fp is not None:
        X_list.append(fp)
        y_list.append(row["expt"])

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Lasso Regression (best alpha = 0.01)
lasso_model = Lasso(alpha=0.01, max_iter=10000)
lasso_model.fit(X_train, y_train)

y_train_pred_lasso = lasso_model.predict(X_train)
y_test_pred_lasso = lasso_model.predict(X_test)

lasso_train_r2 = r2_score(y_train, y_train_pred_lasso)
lasso_test_r2 = r2_score(y_test, y_test_pred_lasso)
lasso_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
lasso_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))
lasso_test_mae = mean_absolute_error(y_test, y_test_pred_lasso)
nonzero_coeffs = np.sum(lasso_model.coef_ != 0)

print("\n LASSO REGRESSION (alpha = 0.01)")
print(f"Training R²: {lasso_train_r2:.4f}")
print(f"Test R²: {lasso_test_r2:.4f}")
print(f"Training RMSE: {lasso_train_rmse:.4f}")
print(f"Test RMSE: {lasso_test_rmse:.4f}")
print(f"Test MAE: {lasso_test_mae:.4f}")
print(f"Number of non-zero coefficients: {nonzero_coeffs}")

# Lasso: Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred_lasso, alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "--", color = 'orange', linewidth=2)
plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("Lasso Regression (alpha = 0.01): Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.show()

# Lasso: Residual plot
lasso_residuals = y_test - y_test_pred_lasso

plt.figure(figsize=(8, 6))
plt.scatter(y_test_pred_lasso, lasso_residuals, alpha=0.7)
plt.axhline(0, linestyle="--", color = 'orange', linewidth=2)
plt.xlabel("Predicted expt")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Lasso Regression (alpha = 0.01): Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# Load dataset
df = pd.read_csv("data/SAMPL.csv")
df = df[["smiles", "expt"]].dropna().reset_index(drop=True)


# Convert SMILES to Morgan fingerprints
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprintAsNumPy(mol)

X_list = []
y_list = []

for _, row in df.iterrows():
    fp = smiles_to_morgan_fp(row["smiles"])
    if fp is not None:
        X_list.append(fp)
        y_list.append(row["expt"])

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# SGD regression without regularisation
sgd_model = SGDRegressor(
    loss="squared_error",
    penalty=None,
    max_iter=5000, #epochs 
    tol=1e-3, #loss drop by less than 10^-3
    random_state=42
)

#values omitted
#learning_rate='invscaling' learning rate decreases over time.
#eta0=0.01 learning rate 
#power_t=0.25 controls how fast the learning rate decreases in the invscaling schedule.
#alpha=0.0001 penalty=None, that alpha is not really active as a regularisation term.
#fit_intercept=True
#shuffle=True
#early_stopping=False
#validation_fraction=0.1
#n_iter_no_change=5

sgd_model.fit(X_train, y_train)

# Predictions
y_train_pred = sgd_model.predict(X_train)
y_test_pred = sgd_model.predict(X_test)

# Metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("\n SGD WITHOUT REGULARISATION ")
print(f"Training R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Training RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Training MAE: {train_mae:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print(f"Number of iterations: {sgd_model.n_iter_}")

# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "--", color = 'orange', linewidth=2)
plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("SGD without Regularisation: Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.show()

# Residual plot
residuals = y_test - y_test_pred

plt.figure(figsize=(8, 6))
plt.scatter(y_test_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--", color = 'orange', linewidth=2)
plt.xlabel("Predicted expt")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("SGD without Regularisation: Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()

# Residual histogram
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, edgecolor="black")
plt.axvline(0, linestyle="--", color = 'orange', linewidth=2)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("SGD without Regularisation: Residual Distribution")
plt.grid(True, alpha=0.3)
plt.show()

## Model comparison: Linear Regression, Ridge, Lasso, and SGD

The four models below were trained to predict experimental solvation free energy (`expt`) from **2048-bit Morgan fingerprints** derived from the SMILES strings.  
The main goal of this comparison is to evaluate **generalisation**, meaning how well each model performs on unseen molecules.

### Comparison table

| Model | Best setting | Train R² | Test R² | Train RMSE | Test RMSE | Notes |
|---|---|---:|---:|---:|---:|---|
| Linear Regression | no regularisation | 0.9995 | 0.7079 | 0.0806 | 2.1993 | Strong overfitting |
| Ridge Regression | alpha = 1.0 | 0.9868 | 0.7517 | 0.4344 | 2.0278 | Best Ridge result |
| Lasso Regression | alpha = 0.01 | 0.8912 | 0.7519 | 1.2452 | 2.0272 | 111 non-zero coefficients |
| SGD Regressor | no regularisation | 0.9573 | 0.7557 | 0.7804 | 2.0114 | Best on this single split (within CV noise — see CV section) |
---

## 1. Linear Regression

Linear Regression is the baseline model. It fits a linear relationship between the fingerprint features and the target without any penalty on the coefficients.

### How to read the results
- The **training R² is almost perfect (0.9995)**.
- The **test R² drops to 0.7079**.
- The **training RMSE is extremely small (0.0806)**, while the **test RMSE is much larger (2.1993)**.

### What this means
This is a very strong sign of **overfitting**.  
The model learned the training data almost perfectly, but it does not generalise well to new molecules.

### Why this happens
The fingerprint matrix has:
- **642 molecules**
- **2048 features**

So the model has a very large number of explanatory variables relative to the amount of data. With ordinary linear regression, there is no mechanism to control coefficient size, so the model can fit noise and dataset-specific patterns.

### Main limitation of this model
It is too flexible for this dataset and becomes overly tailored to the training set.

---

## 2. Ridge Regression

Ridge Regression is a modified linear regression that adds an **L2 penalty** to the coefficients.  
This means the model is still linear, but large coefficients are penalised.

### Best result
The best Ridge model was obtained with **alpha = 1.0**.

### How to read the results
- **Train R² = 0.9868**, slightly lower than Linear Regression
- **Test R² = 0.7517**, clearly higher than Linear Regression
- **Test RMSE = 2.0278**, lower than Linear Regression
- **Test MAE = 1.2173**, lower than Linear Regression

### What changed compared with Linear Regression
Ridge reduces coefficient magnitude, so the model becomes less extreme and less sensitive to noise in the training data.

Compared with ordinary Linear Regression:
- training performance becomes slightly worse
- test performance becomes better

That is actually a **good sign**, because it means the model is sacrificing a little training accuracy in order to improve generalisation.

### Main interpretation
Ridge keeps **all features**, but shrinks their influence.  
This is useful when many fingerprint bits contribute a little and when features may be correlated.

---

## 3. Lasso Regression

Lasso Regression adds an **L1 penalty** instead of an L2 penalty.  
This also shrinks coefficients, but unlike Ridge it can force some coefficients to become exactly zero.

### Best result
The best Lasso model was obtained with **alpha = 0.01**.

### How to read the results
- **Train R² = 0.8912**
- **Test R² = 0.7519**
- **Test RMSE = 2.0272**
- **Test MAE = 1.2578**
- **111 non-zero coefficients**

### What changed compared with Ridge
The test performance is almost the same as Ridge:
- Ridge Test R² = 0.7517
- Lasso Test R² = 0.7519

So both models generalise similarly well.

However, Lasso uses only **111 active features** instead of keeping all 2048.  
This means it performs **feature selection**, removing many coefficients entirely.

### What this means
Lasso produces a **sparser model**, which can be easier to interpret and simpler to describe.

### Important trade-off
Compared with Ridge:
- Lasso is simpler
- but Ridge has a slightly better Test MAE

So Lasso did not clearly outperform Ridge in prediction quality, but it gave a more compact model.

---

## 4. SGD Regressor

SGDRegressor also fits a linear model, but it uses **stochastic gradient descent** instead of solving the problem in closed form.

### Best result
In this case, the best SGD result was obtained **without regularisation**.

### How to read the results
- **Train R² = 0.9573**
- **Test R² = 0.7557**
- **Train RMSE = 0.7804**
- **Test RMSE = 2.0114**
- **Train MAE = 0.5404**
- **Test MAE = 1.2167**

### What changed compared with the other models
Compared with Linear Regression:
- training performance is lower
- test performance is better

Compared with Ridge and Lasso:
- SGD gives the **highest Test R²**
- SGD gives the **lowest Test RMSE**
- SGD gives the **lowest or nearly lowest Test MAE**

On this single 80/20 split, SGD gave the best or nearly best test-set performance. However, this difference is small, so the cross-validation section below is needed to check whether the ranking is robust.

### Why this may happen
Even without explicit regularisation, SGD is an **iterative optimisation method**.  
Because it updates coefficients step by step and stops after a finite number of iterations, it may behave like a softer, more controlled fitting procedure than exact Linear Regression. In practice, this can reduce overfitting.

### Important note
When L2 regularisation was added to SGD, performance became slightly worse.  
This suggests that plain SGD was already giving a good bias–variance balance, and extra shrinkage was not necessary.

---

In [ ]:
# Cross-validation robustness check 
# This checks whether the differences between Linear Regression, Ridge, Lasso and SGD
# are robust or just due to one train/test split.

from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor
from sklearn.metrics import make_scorer, mean_squared_error
import pandas as pd
import numpy as np

def neg_rmse(y_true, y_pred):
    return -np.sqrt(mean_squared_error(y_true, y_pred))

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": make_scorer(neg_rmse),
    "R2": "r2"
}

models_cv = {
    "Linear Regression": LinearRegression(),
    "Ridge alpha=1.0": Ridge(alpha=1.0),
    "Lasso alpha=0.01": Lasso(alpha=0.01, max_iter=20000, random_state=42),
    "SGD no regularisation": SGDRegressor(
        loss="squared_error",
        penalty=None,
        max_iter=10000,
        tol=1e-4,
        random_state=42
    ),
}

cv_rows = []

for name, model_cv in models_cv.items():
    scores = cross_validate(
        model_cv,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise"
    )

    cv_rows.append({
        "Model": name,
        "CV MAE mean": -scores["test_MAE"].mean(),
        "CV MAE std": scores["test_MAE"].std(),
        "CV RMSE mean": -scores["test_RMSE"].mean(),
        "CV RMSE std": scores["test_RMSE"].std(),
        "CV R² mean": scores["test_R2"].mean(),
        "CV R² std": scores["test_R2"].std(),
    })

nb1_cv_results = pd.DataFrame(cv_rows).sort_values("CV MAE mean")
display(nb1_cv_results.round(4))

The comparison shows that the baseline Linear Regression model overfits strongly because the Morgan fingerprint representation is high-dimensional relative to the dataset size. The model uses 2048 fingerprint features for only 642 molecules, so it can fit the training set almost perfectly but generalises less well to unseen molecules.

Ridge and Lasso both reduce overfitting by constraining the model coefficients. Ridge keeps all fingerprint features but shrinks their influence, while Lasso removes many coefficients entirely and produces a sparser model. Both approaches improve test-set performance compared with ordinary Linear Regression.

On the single 80/20 train/test split, SGDRegressor gives the best or nearly best test performance. However, the differences between Ridge, Lasso and SGD are small, so this ranking should not be treated as definitive from one split alone. The cross-validation results provide a more robust estimate of whether these differences are meaningful.

Overall, Notebook 1 should be interpreted as a fingerprint-only baseline on the aqueous SAMPL / FreeSolv subset. Its main conclusion is not that one linear model is definitively best, but that simple Morgan-fingerprint models have limited predictive power and can overfit when the dataset is small. This motivates the move in Notebook 2 to a larger multi-solvent dataset and more chemically informative RDKit descriptors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


#Load dataset

df = pd.read_csv("data/SAMPL.csv")
df = df[["smiles", "expt"]].dropna().reset_index(drop=True)


#Convert SMILES to Morgan fingerprints

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprintAsNumPy(mol)

X_list = []
y_list = []

for _, row in df.iterrows():
    fp = smiles_to_morgan_fp(row["smiles"])
    if fp is not None:
        X_list.append(fp)
        y_list.append(row["expt"])

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)


#Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Linear SVR

svr_linear = make_pipeline(
    StandardScaler(),
    SVR(kernel="linear", C=1.0, epsilon=0.1)
)

svr_linear.fit(X_train, y_train)

# Predictions
y_train_pred_svr = svr_linear.predict(X_train)
y_test_pred_svr = svr_linear.predict(X_test)

# Metrics
svr_train_r2 = r2_score(y_train, y_train_pred_svr)
svr_test_r2 = r2_score(y_test, y_test_pred_svr)
svr_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_svr))
svr_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_svr))
svr_train_mae = mean_absolute_error(y_train, y_train_pred_svr)
svr_test_mae = mean_absolute_error(y_test, y_test_pred_svr)

print("\n===== LINEAR SVR RESULTS =====")
print(f"Train R²: {svr_train_r2:.4f}")
print(f"Test R²: {svr_test_r2:.4f}")
print(f"Train RMSE: {svr_train_rmse:.4f}")
print(f"Test RMSE: {svr_test_rmse:.4f}")
print(f"Train MAE: {svr_train_mae:.4f}")
print(f"Test MAE: {svr_test_mae:.4f}")


#Actual vs Predicted plot

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred_svr, alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "--", linewidth=2)
plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("Linear SVR: Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.show()


#Residual plot

svr_residuals = y_test - y_test_pred_svr

plt.figure(figsize=(8, 6))
plt.scatter(y_test_pred_svr, svr_residuals, alpha=0.7)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("Predicted expt")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Linear SVR: Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()


#Residual histogram

plt.figure(figsize=(8, 6))
plt.hist(svr_residuals, bins=30, edgecolor="black")
plt.axvline(0, linestyle="--", linewidth=2)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Linear SVR: Residual Distribution")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# Load dataset

df = pd.read_csv("data/SAMPL.csv")
df = df[["smiles", "expt"]].dropna().reset_index(drop=True)


# Convert SMILES to Morgan fingerprints

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprintAsNumPy(mol)

X_list = []
y_list = []

for _, row in df.iterrows():
    fp = smiles_to_morgan_fp(row["smiles"])
    if fp is not None:
        X_list.append(fp)
        y_list.append(row["expt"])

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)


# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


#RBF SVR

svr_rbf = make_pipeline(
    StandardScaler(),
    SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale")
)

svr_rbf.fit(X_train, y_train)

# Predictions
y_train_pred_svr_rbf = svr_rbf.predict(X_train)
y_test_pred_svr_rbf = svr_rbf.predict(X_test)

# Metrics
svr_rbf_train_r2 = r2_score(y_train, y_train_pred_svr_rbf)
svr_rbf_test_r2 = r2_score(y_test, y_test_pred_svr_rbf)
svr_rbf_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_svr_rbf))
svr_rbf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_svr_rbf))
svr_rbf_train_mae = mean_absolute_error(y_train, y_train_pred_svr_rbf)
svr_rbf_test_mae = mean_absolute_error(y_test, y_test_pred_svr_rbf)

print("\n===== RBF SVR RESULTS =====")
print(f"Train R²: {svr_rbf_train_r2:.4f}")
print(f"Test R²: {svr_rbf_test_r2:.4f}")
print(f"Train RMSE: {svr_rbf_train_rmse:.4f}")
print(f"Test RMSE: {svr_rbf_test_rmse:.4f}")
print(f"Train MAE: {svr_rbf_train_mae:.4f}")
print(f"Test MAE: {svr_rbf_test_mae:.4f}")


# Actual vs Predicted plot

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred_svr_rbf, alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "--", linewidth=2)
plt.xlabel("Actual expt")
plt.ylabel("Predicted expt")
plt.title("RBF SVR: Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.show()


#Residual plot

svr_rbf_residuals = y_test - y_test_pred_svr_rbf

plt.figure(figsize=(8, 6))
plt.scatter(y_test_pred_svr_rbf, svr_rbf_residuals, alpha=0.7)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("Predicted expt")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("RBF SVR: Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()


#Residual histogram

plt.figure(figsize=(8, 6))
plt.hist(svr_rbf_residuals, bins=30, edgecolor="black")
plt.axvline(0, linestyle="--", linewidth=2)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("RBF SVR: Residual Distribution")
plt.grid(True, alpha=0.3)
plt.show()

Support Vector Regression did not improve the analysis in this case.

The **linear SVR** shows a very high training score but a noticeably lower test score (Train R² = 0.9989 vs Test R² = 0.6563), which suggests strong overfitting. The **RBF SVR** performs even worse on the test set (Test R² = 0.5078, Test RMSE = 2.8550), so introducing a **non-linear kernel** did not help the model generalise to unseen molecules. This indicates that, for this dataset, SVR is less suitable than the regularised linear models, likely because the dataset is relatively small compared with the high-dimensional 2048-bit fingerprint representation. 

In your results, Ridge, Lasso, and SGDRegressor all achieved better test performance than both SVR models, so the regularised linear approaches remain the more appropriate choice here.